# SAGAI v2.1 — Streetscape Analysis with Generative AI

**Version 2.1** — Now uses the official [UVLM package](https://github.com/perezjoan/UVLM) · Apache License 2.0 · [GitHub](https://github.com/perezjoan/SAGAI)

A unified Google Colab notebook for automated streetscape scoring using open-access geospatial data and Vision-Language Models.
SAGAI v2.1 integrates the full [UVLM](https://github.com/perezjoan/UVLM) inference engine, supporting **LLaVA-NeXT** (7 checkpoints) and **Qwen2.5-VL** (4 checkpoints).

---

## How to use this notebook

| Block | What it does | When to run |
|-------|-------------|-------------|
| **Block 0** — Resume Case Study | Mounts Drive, detects existing data, sets case study name | **Instead of** Blocks 1–2 when resuming |
| **Block 1** — OSM Point Generator | Downloads street network, generates sample points | **Once** per study area |
| **Block 2** — Street View Downloader | Batch-downloads Google Street View images | **Once** per study area |
| **Block 3** — VLM Model Loader | Installs dependencies, selects and loads VLM | **Once** per session |
| **Block 4** — Inference Config & Prompts | Defines scoring tasks, prompts, generation params | **Re-run** to change tasks |
| **Block 5** — Batch Scoring Execution | Processes all images, writes results to CSV | **Re-run** for each scoring run |
| **Block 6** — Aggregation & Mapping | Aggregates all indicators to spatial layers, generates maps | **Re-run** after scoring |

### Key features (v2.1)

- **Single notebook** — All 4 modules unified into one streamlined workflow
- **Resume support** — Block 0 detects existing data on Drive, skip straight to scoring or mapping
- **Widget-based configuration** — Dropdown menus and interactive forms for every block
- **Flexible image modes** — Download 4 cardinal views, left+right sides, or front+back views
- **UVLM-powered inference** — 11 VLM checkpoints (LLaVA-NeXT + Qwen2.5-VL) with dual-backend routing
- **Multi-task prompting** — Up to 10 scoring tasks per run (numeric, category, boolean, text)
- **Consensus validation** — Majority voting across repeated inferences
- **Advanced reasoning** — Chain-of-thought prompting with structured answer extraction
- **Full spatial join** — All indicators and per-view scores joined to the output GeoPackage regardless of mapping selection
- **Adaptive mapping** — Automatically adjusts aggregation to match the image mode used

> **Citation**: Perez, J. and Fusco, G. (2025). *Streetscape Analysis with Generative AI (SAGAI)*. Geomatica 77(2), 100063.


In [1]:
# ===============================================
# BLOCK 0 -- Resume an Existing Case Study
# ===============================================
# Run this block INSTEAD of Blocks 1-2 when your points
# and/or images already exist on Drive from a previous session.
# It sets the global case_study_name variable so that
# Blocks 3-6 work without re-running the full pipeline.
#
# Points + images exist  -> skip to Block 3
# Points + images + CSV  -> skip to Block 6

from google.colab import drive
import os
import ipywidgets as widgets
from IPython.display import display, clear_output

drive.mount("/content/drive")

_sagai_base = "/content/drive/MyDrive/SAGAI"
_samples_dir = os.path.join(_sagai_base, "StreetSamples")

_found = []
if os.path.isdir(_samples_dir):
    for f in sorted(os.listdir(_samples_dir)):
        if f.endswith("_osm.gpkg"):
            name = f.replace("_osm.gpkg", "")
            _found.append(name)

if not _found:
    print("No existing case studies found in /content/drive/MyDrive/SAGAI/StreetSamples/")
    print("Run Block 1 to create a new one.")
else:
    _cs_dropdown = widgets.Dropdown(
        options=_found,
        value=_found[0],
        description="Case study",
        layout=widgets.Layout(width="50%"),
    )

    _resume_btn = widgets.Button(
        description="Resume this case study",
        button_style="success",
        icon="play",
        layout=widgets.Layout(width="250px"),
    )

    _resume_output = widgets.Output()

    def _on_resume(b):
        global case_study_name

        with _resume_output:
            clear_output(wait=True)
            case_study_name = _cs_dropdown.value

            gpkg_path = os.path.join(_samples_dir, f"{case_study_name}_osm.gpkg")
            dl_dir = os.path.join(_sagai_base,
                        f"StreetViewBatchDownload_{case_study_name.capitalize()}")

            has_gpkg = os.path.exists(gpkg_path)
            has_images = os.path.isdir(dl_dir) and any(
                fn.endswith(".jpg") for fn in os.listdir(dl_dir))
            n_images = len([fn for fn in os.listdir(dl_dir) if fn.endswith(".jpg")]) if has_images else 0

            csv_path = None
            for search_dir in [dl_dir, os.path.join(_sagai_base, "Image_Analysis")]:
                if os.path.isdir(search_dir):
                    candidates = [fn for fn in os.listdir(search_dir)
                                  if fn.startswith("Score_Analysis") and fn.endswith(".csv")]
                    if candidates:
                        csv_path = os.path.join(search_dir, sorted(candidates)[0])
                        break

            print(f"[OK] case_study_name = '{case_study_name}'")
            print()
            gpkg_status = "[OK] " + gpkg_path if has_gpkg else "[X] not found"
            img_status = "[OK] " + str(n_images) + " JPGs in " + dl_dir if has_images else "[X] not found"
            csv_status = "[OK] " + csv_path if csv_path else "[ ] not yet (run Blocks 3-5)"
            print(f"   GeoPackage:  {gpkg_status}")
            print(f"   Images:      {img_status}")
            print(f"   Analysis:    {csv_status}")
            print()

            if has_gpkg and has_images and csv_path:
                print(">> Everything exists -- you can skip to Block 6 directly.")
            elif has_gpkg and has_images:
                print(">> Points + images exist -- skip to Block 3 (load VLM) then 4 > 5 > 6.")
            elif has_gpkg:
                print(">> Points exist but no images -- skip to Block 2 then continue normally.")
            else:
                print(">> GeoPackage not found -- you may need to run Block 1.")

    _resume_btn.on_click(_on_resume)

    display(widgets.VBox([
        widgets.HTML("<h3>Block 0 -- Resume Existing Case Study</h3>"),
        widgets.HTML(
            "<p>Select an existing case study from Drive to resume "
            "without re-running Blocks 1-2.</p>"
        ),
        _cs_dropdown,
        _resume_btn,
        _resume_output,
    ]))


Mounted at /content/drive


In [ ]:
# ===============================================
# BLOCK 1 — OSM Point Generator (Interactive Map)
# ===============================================
# Draw a rectangle or polygon on the interactive map, or load
# a polygon from a file (GeoPackage, GeoJSON, Shapefile).
# Points are generated along the OSM street network.
# The projected CRS is auto-detected (UTM zone) for worldwide use.
# Highway type filtering removes sidewalks/paths to avoid duplicates.
# The study area polygon can be saved for reproducibility.

!pip install -q osmnx ipyleaflet

import os
import math
import json as _json
import geopandas as gpd
import osmnx as ox
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import box, LineString, Point, Polygon, shape, mapping
from shapely.ops import unary_union
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# ------------------------------------------------
# Interactive Map with Drawing Controls
# ------------------------------------------------
from ipyleaflet import Map, DrawControl, GeoJSON, basemaps, WidgetControl
import ipywidgets as widgets

# State: store the drawn geometry
DRAWN_GEOMETRY = {"geom": None, "type": None}

# Map centered on Europe by default
m = Map(
    basemap={
        "url": "https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png",
        "max_zoom": 19,
        "attribution": "CartoDB",
        "name": "CartoDB Positron",
    },
    center=(46.5, 6.5),
    zoom=5,
    layout=widgets.Layout(width='100%', height='500px'),
)

# Drawing controls: rectangle + polygon only
draw_control = DrawControl(
    rectangle={"shapeOptions": {"color": "#e74c3c", "fillOpacity": 0.15, "weight": 2}},
    polygon={"shapeOptions": {"color": "#3498db", "fillOpacity": 0.15, "weight": 2}},
    circle={},
    circlemarker={},
    polyline={},
    marker={},
)

draw_status = widgets.HTML(
    value='<b style="color:#888;">Draw a polygon on the map or load one from file.</b>'
)

def _update_status(geom, source="drawn"):
    """Update the status widget with geometry info."""
    bounds = geom.bounds
    area_approx = abs(bounds[2] - bounds[0]) * abs(bounds[3] - bounds[1]) * 111**2
    DRAWN_GEOMETRY["geom"] = geom
    DRAWN_GEOMETRY["type"] = geom.geom_type
    draw_status.value = (
        f'<b style="color:#27ae60;">{source.capitalize()} {geom.geom_type}!</b> '
        f'Bounds: [{bounds[0]:.4f}, {bounds[1]:.4f}] to [{bounds[2]:.4f}, {bounds[3]:.4f}] '
        f'(~{area_approx:.1f} km\u00b2)'
    )

def handle_draw(self, action, geo_json):
    """Callback: capture drawn geometry."""
    if action == 'created':
        geom_type = geo_json['geometry']['type']
        coords = geo_json['geometry']['coordinates']
        if geom_type == 'Polygon':
            geom = Polygon(coords[0])
        else:
            geom = shape(geo_json['geometry'])
        _update_status(geom, "Drawn")

draw_control.on_draw(handle_draw)
m.add(draw_control)

status_control = WidgetControl(widget=draw_status, position='bottomleft')
m.add(status_control)

# Track loaded layers so we can remove them
_loaded_layers = []

# ------------------------------------------------
# Load polygon from file
# ------------------------------------------------
polygon_path_widget = widgets.Text(
    value='',
    description='Polygon file',
    placeholder='/content/drive/MyDrive/study_area.gpkg',
    layout=widgets.Layout(width='70%'),
)

load_polygon_file_btn = widgets.Button(
    description='Load Polygon',
    button_style='info',
    icon='upload',
    layout=widgets.Layout(width='150px'),
)

load_output = widgets.Output()

def on_load_polygon(b):
    with load_output:
        clear_output(wait=True)
        file_path = polygon_path_widget.value.strip()
        if not file_path:
            print("Enter a file path (e.g., /content/drive/MyDrive/study_area.gpkg)")
            return
        if not os.path.exists(file_path):
            print(f"File not found: {file_path}")
            return
        try:
            name = os.path.basename(file_path)

            # Read with geopandas
            gdf = gpd.read_file(file_path)

            # Reproject to WGS84 if needed
            if gdf.crs and gdf.crs != "EPSG:4326":
                gdf = gdf.to_crs("EPSG:4326")

            # Filter out empty/null geometries
            gdf = gdf[~gdf.geometry.is_empty & gdf.geometry.notna()].copy()
            if len(gdf) == 0:
                print("No valid geometries found in file.")
                return

            # Union all geometries into one polygon
            merged = unary_union(gdf.geometry)
            if merged.geom_type == 'MultiPolygon':
                # Take the largest polygon
                merged = max(merged.geoms, key=lambda g: g.area)

            _update_status(merged, "Loaded")

            # Display on map
            for layer in _loaded_layers:
                try:
                    m.remove(layer)
                except:
                    pass
            _loaded_layers.clear()

            geo_layer = GeoJSON(
                data={"type": "Feature", "geometry": mapping(merged)},
                style={"color": "#9b59b6", "fillOpacity": 0.12, "weight": 2},
            )
            m.add(geo_layer)
            _loaded_layers.append(geo_layer)

            # Center map on loaded polygon
            bounds = merged.bounds
            center_lat = (bounds[1] + bounds[3]) / 2
            center_lon = (bounds[0] + bounds[2]) / 2
            m.center = (center_lat, center_lon)
            m.zoom = 13

            print(f"Loaded: {name} ({len(gdf)} features -> 1 polygon)")

        except Exception as e:
            print(f"Error loading file: {e}")

load_polygon_file_btn.on_click(on_load_polygon)

# ------------------------------------------------
# Save study area polygon
# ------------------------------------------------
save_polygon_button = widgets.Button(
    description='Save Study Area',
    button_style='warning',
    icon='save',
    layout=widgets.Layout(width='200px'),
)

def on_save_polygon(b):
    with load_output:
        clear_output(wait=True)
        if DRAWN_GEOMETRY["geom"] is None:
            print("No polygon to save. Draw or load one first.")
            return
        cs = case_study_widget.value.strip()
        if not cs:
            print("Enter a case study name first.")
            return
        base_dir = "/content/drive/MyDrive/SAGAI"
        samples_dir = os.path.join(base_dir, "StreetSamples")
        os.makedirs(samples_dir, exist_ok=True)
        save_path = os.path.join(samples_dir, f"{cs}_study_area.gpkg")
        poly_gdf = gpd.GeoDataFrame(
            {"case_study": [cs]},
            geometry=[DRAWN_GEOMETRY["geom"]],
            crs="EPSG:4326"
        )
        poly_gdf.to_file(save_path, driver="GPKG")
        print(f"Study area saved: {save_path}")

save_polygon_button.on_click(on_save_polygon)

# ------------------------------------------------
# Configuration Widgets
# ------------------------------------------------
case_study_widget = widgets.Text(
    value='',
    description='Case study',
    placeholder='e.g., nice, vienna, tokyo',
    layout=widgets.Layout(width='50%'),
)

spacing_widget = widgets.IntSlider(
    value=40, min=10, max=200, step=5,
    description='Spacing (m)',
    style={'description_width': 'initial'},
)

offset_widget = widgets.IntSlider(
    value=15, min=5, max=50, step=5,
    description='Offset (m)',
    style={'description_width': 'initial'},
)

# Highway type filter
highway_filter_widget = widgets.SelectMultiple(
    options=[
        'All (no filter)',
        'motorway', 'motorway_link',
        'trunk', 'trunk_link',
        'primary', 'primary_link',
        'secondary', 'secondary_link',
        'tertiary', 'tertiary_link',
        'residential', 'living_street',
        'unclassified', 'service',
        'pedestrian', 'footway', 'path',
        'cycleway', 'steps', 'track',
    ],
    value=['All (no filter)'],
    description='Highway types',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='50%', height='130px'),
)

highway_filter_help = widgets.HTML(
    value="<small><i>"
          "Filter OSM street types to include. Select <b>All (no filter)</b> to keep everything.<br>"
          "To avoid sidewalk duplicates, deselect <code>footway</code>, <code>path</code>, "
          "<code>cycleway</code>, <code>steps</code>.<br>"
          "Hold Ctrl/Cmd to select multiple types."
          "</i></small>"
)

crs_info = widgets.HTML(
    value='<small><i>Projected CRS will be auto-detected (UTM zone) from your study area centroid. '
          'Works worldwide with optimal precision.</i></small>'
)

run_block1_button = widgets.Button(
    description='Generate Points',
    button_style='success',
    icon='globe',
)

block1_output = widgets.Output()

# ------------------------------------------------
# Core algorithm callback
# ------------------------------------------------
def run_block1(b):
    global points_gdf, streets_gdf, save_path_gpkg, case_study_name

    with block1_output:
        clear_output(wait=True)

        if DRAWN_GEOMETRY["geom"] is None:
            print("Please draw a polygon on the map or load one from file.")
            return

        print("Processing...")

        try:
            case_study_name = case_study_widget.value.strip()
            if not case_study_name:
                print("Please enter a case study name.")
                return

            point_distance = spacing_widget.value
            offset_distance = offset_widget.value
            study_polygon = DRAWN_GEOMETRY["geom"]
            highway_selection = list(highway_filter_widget.value)

            # Create GeoDataFrame from drawn geometry
            polygon_gdf = gpd.GeoDataFrame(
                geometry=[study_polygon], crs="EPSG:4326"
            )

            # --- Auto-detect projected CRS (UTM zone) ---
            projected_crs = polygon_gdf.estimate_utm_crs()
            print(f"Auto-detected CRS: {projected_crs} (from centroid)")

            # Folder structure
            base_dir = "/content/drive/MyDrive/SAGAI"
            samples_dir = os.path.join(base_dir, "StreetSamples")
            os.makedirs(samples_dir, exist_ok=True)
            save_path_gpkg = os.path.join(samples_dir, f"{case_study_name}_osm.gpkg")

            # Save study area polygon for reproducibility
            poly_save = os.path.join(samples_dir, f"{case_study_name}_study_area.gpkg")
            polygon_gdf.to_file(poly_save, driver="GPKG")
            print(f"Study area saved: {poly_save}")

            # Download OSM streets within the drawn polygon
            ox.settings.timeout = 180
            ox.settings.overpass_rate_limit = True

            import time as _time
            t0 = _time.time()
            print("Downloading OSM streets from Overpass API...")
            print("   (this may take 30s-3min depending on area size and server load)")

            try:
                G = ox.graph_from_polygon(study_polygon, network_type='all')
            except Exception as e:
                elapsed = _time.time() - t0
                print(f"   Failed after {elapsed:.0f}s: {e}")
                print("   Possible causes:")
                print("   - Overpass API overloaded (try again in 1-2 min)")
                print("   - Study area too large (reduce polygon)")
                print("   - Network issue (check internet connection)")
                print(f"   Check server status: https://overpass-api.de/api/status")
                raise

            elapsed = _time.time() - t0
            print(f"   Downloaded in {elapsed:.0f}s "
                  f"({len(G.nodes)} nodes, {len(G.edges)} edges)")

            nodes, edges = ox.graph_to_gdfs(G)
            streets_gdf = edges.copy()

            # Convert list attributes to strings
            def convert_list_to_string(val):
                return ', '.join(map(str, val)) if isinstance(val, list) else val
            for column in streets_gdf.columns:
                if any(isinstance(val, list) for val in streets_gdf[column]):
                    streets_gdf[column] = streets_gdf[column].apply(convert_list_to_string)

            # --- Highway type filter ---
            n_before_filter = len(streets_gdf)
            if 'All (no filter)' not in highway_selection:
                if 'highway' in streets_gdf.columns:
                    streets_gdf = streets_gdf[
                        streets_gdf['highway'].isin(highway_selection)
                    ].copy()
                    print(f"Highway filter: {n_before_filter} -> {len(streets_gdf)} streets "
                          f"(kept: {', '.join(highway_selection)})")
                else:
                    print("Warning: 'highway' column not found, no filter applied.")
            else:
                print(f"No highway filter applied ({n_before_filter} streets)")

            # Reproject to projected CRS for distance calculations
            streets_gdf = streets_gdf.to_crs(projected_crs)
            polygon_proj = polygon_gdf.to_crs(projected_crs)
            bbox_area_km2 = polygon_proj.geometry.area.iloc[0] / 1e6

            # Calculate lengths
            streets_gdf["length_m"] = streets_gdf.geometry.length
            total_length_km = streets_gdf["length_m"].sum() / 1000

            # Remove duplicates
            if 'osmid' in streets_gdf.columns:
                streets_gdf['length_m'] = streets_gdf['length_m'].round(2)
                streets_gdf = streets_gdf.drop_duplicates(
                    subset=['osmid', 'length_m'], keep='first'
                )

            # Keep essential columns + highway tag
            keep_cols = ['geometry', 'length_m', 'osmid', 'highway', 'name']
            streets_gdf = streets_gdf[[c for c in keep_cols if c in streets_gdf.columns]]
            streets_gdf = streets_gdf.reset_index(drop=True)
            streets_gdf['street_id'] = [f"street_{i+1}" for i in range(len(streets_gdf))]

            # Generate points along streets
            print("Generating points...")
            all_points = []
            point_counter = 1
            for idx, row in streets_gdf.iterrows():
                line = row.geometry
                length = line.length
                sid = row['street_id']

                if length < offset_distance:
                    continue
                elif length < offset_distance + point_distance:
                    pt = line.interpolate(length / 2)
                    all_points.append({
                        'geometry': pt,
                        'point_id': f"point_{point_counter}",
                        'street_id': sid,
                    })
                    point_counter += 1
                else:
                    start = offset_distance
                    end = length - offset_distance
                    current = start
                    while current <= end:
                        pt = line.interpolate(current)
                        all_points.append({
                            'geometry': pt,
                            'point_id': f"point_{point_counter}",
                            'street_id': sid,
                        })
                        point_counter += 1
                        current += point_distance

            points_gdf = gpd.GeoDataFrame(all_points, crs=projected_crs)

            # Convert to WGS84 for lat/lon columns
            points_wgs = points_gdf.to_crs("EPSG:4326")
            points_gdf['Latitude'] = points_wgs.geometry.y
            points_gdf['Longitude'] = points_wgs.geometry.x

            # Save as GeoPackage
            streets_out = streets_gdf.to_crs(epsg=4326)
            points_out = points_gdf.to_crs(epsg=4326)
            streets_out.to_file(save_path_gpkg, layer="streets", driver="GPKG")
            points_out.to_file(save_path_gpkg, layer="points", driver="GPKG")

            # Visualization
            fig, ax = plt.subplots(figsize=(10, 10))
            polygon_gdf.boundary.plot(ax=ax, color='blue', linewidth=2, linestyle='--', label='Study area')
            streets_out.plot(ax=ax, color='grey', linewidth=0.5, label='Streets')
            points_out.plot(ax=ax, color='red', markersize=5, label='Points')
            ax.set_title(f'{case_study_name.capitalize()} \u2014 Streets & Sample Points')
            ax.legend()
            ax.grid(True)
            ax.tick_params(axis='x', rotation=45)
            plt.tight_layout()
            plt.show()

            # Summary
            print(f"\nPoints generated successfully!")
            print(f"   Study area: {DRAWN_GEOMETRY['type']} (~{bbox_area_km2:.2f} km\u00b2)")
            print(f"   Projected CRS: {projected_crs}")
            print(f"   Streets: {len(streets_gdf)} | Points: {len(points_gdf)}")
            print(f"   Total street length: {total_length_km:.2f} km")
            print(f"   Saved to: {save_path_gpkg}")

        except Exception as e:
            print(f"Error: {e}")
            raise

run_block1_button.on_click(run_block1)

# ------------------------------------------------
# Display UI
# ------------------------------------------------
display(widgets.VBox([
    widgets.HTML('<h3>Block 1 \u2014 OSM Point Generator</h3>'),
    widgets.HTML(
        '<p>Draw a <b>rectangle</b> or <b>polygon</b> on the map, '
        'or <b>load</b> one from a GeoPackage/GeoJSON/Shapefile. '
        'Then configure parameters and click <b>Generate Points</b>.</p>'
    ),
    m,
    widgets.HTML('<br>'),
    widgets.HBox([polygon_path_widget, load_polygon_file_btn, save_polygon_button]),
    load_output,
    widgets.HTML('<br>'),
    case_study_widget,
    spacing_widget,
    offset_widget,
    highway_filter_widget,
    highway_filter_help,
    crs_info,
    run_block1_button,
    block1_output,
]))


In [ ]:
# ===============================================
# BLOCK 2 — Street View Batch Downloader (v2.1)
# ===============================================
# Pipeline:
# 1. Compute canonical bearings on original on-street points
# 2. Query free Metadata API for each point
# 3. Filter out indoor/contributor 360 photos (non-Google)
# 4. Snap points to real camera locations
# 5. Remove points beyond snap threshold
# 6. Deduplicate by pano_id (same panorama = same images)
# 7. Download images only for surviving unique points
#
# Bearing: Canonical bearing normalized to [0,180) so that
# "left" and "right" are geographically consistent regardless
# of OSM street digitization direction.

import os
import math
import time
import requests
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from PIL import Image as PILImage
from io import BytesIO
from shapely.geometry import LineString, Point
from shapely.ops import substring
import ipywidgets as widgets
from IPython.display import display, clear_output

# ------------------------------------------------
# UI Widgets (no case study — uses global from Block 1)
# ------------------------------------------------
api_key_widget = widgets.Text(
    value='',
    description='API Key',
    placeholder='Your Google Maps API key',
    layout=widgets.Layout(width='70%'),
)

image_mode_widget = widgets.Dropdown(
    options=[
        ('4 views: left + right + front + back', '4_views'),
        ('Left + Right sides only', 'left_right'),
        ('Front + Back views only', 'front_back'),
    ],
    value='4_views',
    description='Image mode',
    layout=widgets.Layout(width='60%'),
)

image_mode_help = widgets.HTML(
    value="<small><i>"
          "<b>4 views</b>: Left, right, front, back "
          "(perpendicular + along street) — full coverage<br>"
          "<b>Left + Right</b>: Perpendicular to street axis "
          "— ideal for facade analysis<br>"
          "<b>Front + Back</b>: Along street axis "
          "— ideal for corridor analysis<br>"
          "Left/right are geographically consistent "
          "(canonical bearing, independent of OSM digitization)."
          "</i></small>"
)

pitch_widget = widgets.IntSlider(value=-20, min=-90, max=90, step=5, description='Pitch')
fov_widget = widgets.IntSlider(value=90, min=10, max=120, step=10, description='FOV')

max_sub_widget = widgets.IntSlider(
    value=50, min=10, max=200, step=5,
    description='Sub-seg (m)',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='50%'),
)

max_sub_help = widgets.HTML(
    value="<small><i>"
          "Max sub-segment length for bearing computation.<br>"
          "Winding roads: <b>10–25 m</b> | "
          "Mixed suburban: <b>40–60 m</b> | "
          "Straight grid: <b>80–200 m</b>"
          "</i></small>"
)

snap_threshold_widget = widgets.IntSlider(
    value=30, min=5, max=100, step=5,
    description='Snap max (m)',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='50%'),
)

snap_help = widgets.HTML(
    value="<small><i>"
          "Max distance (m) between generated point and nearest "
          "Street View camera. Points farther are skipped."
          "</i></small>"
)

run_block2_button = widgets.Button(description='Download Images', button_style='success')
block2_output = widgets.Output()

# Progress
progress_bar = widgets.IntProgress(
    value=0, min=0, max=100,
    description='Progress:',
    bar_style='info',
    layout=widgets.Layout(width='80%'),
)
progress_label = widgets.HTML(value='')
progress_box = widgets.VBox([progress_bar, progress_label],
                            layout=widgets.Layout(display='none'))

# ------------------------------------------------
# Canonical bearing: normalize to [0, 180)
# ------------------------------------------------
def canonical_bearing(raw_bearing):
    """
    Normalize bearing to [0, 180).
    A street digitized S->N (bearing ~0) and N->S (bearing ~180)
    both become ~0. This ensures 'left' is always the same
    geographic side regardless of OSM digitization direction.
    """
    b = raw_bearing % 360
    if b >= 180:
        b -= 180
    return b

# ------------------------------------------------
# Sub-segment bearing computation (Node-to-Node)
# ------------------------------------------------
def subdivide_segment(line_geom, max_length=50.0):
    total = line_geom.length
    if total <= max_length:
        return [line_geom]
    sub_segments = []
    n_parts = math.ceil(total / max_length)
    part_length = total / n_parts
    for i in range(n_parts):
        start_dist = i * part_length
        end_dist = min((i + 1) * part_length, total)
        sub = substring(line_geom, start_dist, end_dist)
        if sub is not None and not sub.is_empty and sub.length > 0:
            sub_segments.append(sub)
    return sub_segments

def node_to_node_bearing(line_geom):
    coords = list(line_geom.coords)
    if len(coords) < 2:
        return 0.0
    dx = coords[-1][0] - coords[0][0]
    dy = coords[-1][1] - coords[0][1]
    return math.degrees(math.atan2(dx, dy)) % 360

def find_sub_segment_for_point(point_geom, sub_segments):
    best_seg = sub_segments[0]
    best_dist = float('inf')
    for seg in sub_segments:
        d = seg.distance(point_geom)
        if d < best_dist:
            best_dist = d
            best_seg = seg
    return best_seg

def compute_bearing_for_point(point_geom, street_geom, max_sub_length=50.0):
    sub_segs = subdivide_segment(street_geom, max_sub_length)
    best_seg = find_sub_segment_for_point(point_geom, sub_segs)
    raw = node_to_node_bearing(best_seg)
    return canonical_bearing(raw)

def get_headings_for_mode(mode, bearing=0.0):
    """
    Return (heading, suffix) tuples.
    bearing is canonical [0,180) = the 'forward' direction of the street.
    left  = bearing - 90 (always same geographic side)
    right = bearing + 90
    front = bearing
    back  = bearing + 180
    """
    front = bearing % 360
    back = (bearing + 180) % 360
    left = (bearing + 270) % 360
    right = (bearing + 90) % 360

    if mode == '4_views':
        return [(left, 'left'), (right, 'right'),
                (front, 'front'), (back, 'back')]
    elif mode == 'left_right':
        return [(left, 'left'), (right, 'right')]
    elif mode == 'front_back':
        return [(front, 'front'), (back, 'back')]
    return [(left, 'left'), (right, 'right'),
            (front, 'front'), (back, 'back')]

# ------------------------------------------------
# Metadata API
# ------------------------------------------------
def query_sv_metadata(lat, lon, api_key, radius=50):
    url = (f"https://maps.googleapis.com/maps/api/streetview/metadata"
           f"?location={lat},{lon}&radius={radius}&key={api_key}")
    try:
        r = requests.get(url, timeout=10)
        data = r.json()
        if data.get('status') == 'OK':
            loc = data['location']
            return {
                'status': 'OK',
                'pano_id': data.get('pano_id', ''),
                'lat': loc['lat'],
                'lon': loc['lng'],
                'copyright': data.get('copyright', ''),
            }
        else:
            return {'status': data.get('status', 'UNKNOWN'),
                    'pano_id': '', 'lat': lat, 'lon': lon,
                    'copyright': ''}
    except Exception:
        return {'status': 'ERROR', 'pano_id': '', 'lat': lat,
                'lon': lon, 'copyright': ''}

# ------------------------------------------------
# Download function
# ------------------------------------------------
def download_streetview(lat, lon, heading, pitch, fov, save_path,
                        filename, api_key, pano_id=None):
    if pano_id:
        url = (f"https://maps.googleapis.com/maps/api/streetview"
               f"?size=640x640&pano={pano_id}"
               f"&heading={heading}&pitch={pitch}&fov={fov}&key={api_key}")
    else:
        url = (f"https://maps.googleapis.com/maps/api/streetview"
               f"?size=640x640&location={lat},{lon}"
               f"&heading={heading}&pitch={pitch}&fov={fov}&key={api_key}")
    response = requests.get(url, stream=True)
    if response.status_code == 200:
        image = PILImage.open(BytesIO(response.content)).convert("RGB")
        image_array = np.array(image)
        unique, counts = np.unique(
            image_array.reshape(-1, 3), axis=0, return_counts=True
        )
        most_common_ratio = counts.max() / counts.sum()
        is_blank = most_common_ratio > 0.9
        suffix = "_NA" if is_blank else ""
        file_name = f"{filename}{suffix}.jpg"
        file_path = os.path.join(save_path, file_name)
        image.save(file_path)
        return is_blank
    return True

# ------------------------------------------------
# Coverage map
# ------------------------------------------------
def plot_coverage_map(gpkg_path, download_dir, case_study, mode, n_per_point):
    points_gdf_map = gpd.read_file(gpkg_path, layer='points')
    streets_gdf_map = gpd.read_file(gpkg_path, layer='streets')

    if mode == 'left_right':
        suffixes = ['left', 'right']
    elif mode == 'front_back':
        suffixes = ['front', 'back']
    else:
        suffixes = ['left', 'right', 'front', 'back']

    coverage = []
    for _, row in points_gdf_map.iterrows():
        pid = row['point_id']
        n_valid = 0
        for s in suffixes:
            if os.path.exists(os.path.join(download_dir, f"{pid}_{s}.jpg")):
                n_valid += 1
        coverage.append(n_valid)

    points_gdf_map['n_valid_images'] = coverage
    points_gdf_map['has_coverage'] = points_gdf_map['n_valid_images'] > 0
    points_gdf_map = points_gdf_map.to_crs("EPSG:4326")
    streets_gdf_map = streets_gdf_map.to_crs("EPSG:4326")

    fig, ax = plt.subplots(figsize=(12, 12))
    streets_gdf_map.plot(ax=ax, color='#cccccc', linewidth=0.5, zorder=1)
    has_img = points_gdf_map[points_gdf_map['has_coverage']]
    no_img = points_gdf_map[~points_gdf_map['has_coverage']]
    if len(has_img) > 0:
        has_img.plot(ax=ax, color='#2ecc71', markersize=15,
                     label=f'With images ({len(has_img)})', zorder=2, alpha=0.8)
    if len(no_img) > 0:
        no_img.plot(ax=ax, color='#e74c3c', markersize=15,
                    label=f'No images ({len(no_img)})', zorder=3, alpha=0.8)

    mode_label = {'left_right': 'left+right', 'front_back': 'front+back',
                  '4_views': '4 views'}.get(mode, mode)
    total = len(points_gdf_map)
    pct = len(has_img) / total * 100 if total > 0 else 0
    ax.set_title(f'{case_study.capitalize()} — Coverage ({mode_label})\n'
                 f'{len(has_img)}/{total} points ({pct:.1f}%)', fontsize=14)
    ax.legend(loc='lower right', fontsize=11, frameon=True)
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
    ax.tick_params(axis='x', rotation=45); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()
    return len(has_img), len(no_img)

# ------------------------------------------------
# Main callback
# ------------------------------------------------
def run_block2(b):
    global download_dir, image_mode_used

    with block2_output:
        clear_output(wait=True)

        try:
            case_study = case_study_name  # from Block 1
            api_key = api_key_widget.value.strip()
            mode = image_mode_widget.value
            pitch = pitch_widget.value
            fov = fov_widget.value
            max_sub = max_sub_widget.value
            snap_max_m = snap_threshold_widget.value
            image_mode_used = mode

            if not api_key:
                print("Please enter your Google Maps API key.")
                return

            base_dir = '/content/drive/MyDrive/SAGAI'
            sample_dir = os.path.join(base_dir, 'StreetSamples')
            download_dir = os.path.join(base_dir,
                           f'StreetViewBatchDownload_{case_study.capitalize()}')
            os.makedirs(download_dir, exist_ok=True)
            gpkg_path = os.path.join(sample_dir, f'{case_study}_osm.gpkg')

            points_gdf_local = gpd.read_file(gpkg_path, layer='points')
            n_original = len(points_gdf_local)

            print(f"Case study: {case_study}")
            print(f"Image mode: {mode}")

            # ==============================================
            # PHASE 1: Compute bearings on ORIGINAL positions
            # ==============================================
            # IMPORTANT: bearings must be computed BEFORE snapping
            # because original points sit exactly on the street
            # geometry. After snapping, points move to camera
            # locations which may be off-street, giving wrong bearings.
            point_bearings = {}
            streets_gdf_local = gpd.read_file(gpkg_path, layer='streets')
            proj_crs = points_gdf_local.estimate_utm_crs()
            points_proj = points_gdf_local.to_crs(proj_crs)
            streets_proj = streets_gdf_local.to_crs(proj_crs)

            print(f"\nPhase 1: Computing canonical bearings "
                  f"(sub-seg: {max_sub}m)...")

            street_geom_lookup = dict(
                zip(streets_proj['street_id'], streets_proj.geometry)
            )
            for idx, row in points_proj.iterrows():
                pid = row['point_id']
                sid = row.get('street_id', None)
                if sid and sid in street_geom_lookup:
                    bearing = compute_bearing_for_point(
                        row.geometry, street_geom_lookup[sid],
                        max_sub_length=float(max_sub)
                    )
                    point_bearings[pid] = bearing
                else:
                    point_bearings[pid] = 0.0

            print(f"   Bearings computed for {len(point_bearings)} points "
                  f"(from original on-street positions)")

            # ==============================================
            # PHASE 2: Metadata query + filter + snap + dedup
            # ==============================================
            print(f"\nPhase 2: Querying metadata for {n_original} points...")
            progress_box.layout.display = ''
            progress_bar.max = n_original
            progress_bar.value = 0
            progress_bar.bar_style = 'info'
            progress_bar.description = 'Metadata:'

            metadata_results = []
            for i, (idx, row) in enumerate(points_gdf_local.iterrows()):
                lat = row['Latitude']
                lon = row['Longitude']
                meta = query_sv_metadata(lat, lon, api_key, radius=snap_max_m)
                meta['point_id'] = row['point_id']
                meta['street_id'] = row['street_id']
                meta['orig_lat'] = lat
                meta['orig_lon'] = lon
                metadata_results.append(meta)
                progress_bar.value = i + 1
                if (i + 1) % 50 == 0 or (i + 1) == n_original:
                    progress_label.value = (
                        f'<small>Metadata: {i+1}/{n_original} '
                        f'({(i+1)/n_original*100:.0f}%)</small>'
                    )

            meta_df = pd.DataFrame(metadata_results)

            # Filter: keep only OK status
            ok_mask = meta_df['status'] == 'OK'
            n_no_coverage = (~ok_mask).sum()
            meta_df = meta_df[ok_mask].copy()

            # Filter: remove indoor/contributor 360 photos
            n_before_indoor = len(meta_df)
            google_mask = meta_df['copyright'].str.contains(
                'Google', case=False, na=False
            )
            meta_df = meta_df[google_mask].copy()
            n_indoor = n_before_indoor - len(meta_df)

            # Compute snap distance
            from math import radians, sin, cos, sqrt, atan2
            def haversine_m(lat1, lon1, lat2, lon2):
                R = 6371000
                dlat = radians(lat2 - lat1)
                dlon = radians(lon2 - lon1)
                a = (sin(dlat/2)**2 +
                     cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2)
                return R * 2 * atan2(sqrt(a), sqrt(1-a))

            meta_df['snap_dist_m'] = meta_df.apply(
                lambda r: haversine_m(
                    r['orig_lat'], r['orig_lon'], r['lat'], r['lon']
                ), axis=1
            )

            # Filter: snap distance
            n_before_snap = len(meta_df)
            meta_df = meta_df[meta_df['snap_dist_m'] <= snap_max_m].copy()
            n_too_far = n_before_snap - len(meta_df)

            # Deduplicate by pano_id
            n_before_dedup = len(meta_df)
            meta_df = meta_df.sort_values('snap_dist_m').drop_duplicates(
                subset='pano_id', keep='first'
            ).copy()
            n_deduped = n_before_dedup - len(meta_df)

            print(f"\n   Original points:       {n_original}")
            print(f"   No Street View:        {n_no_coverage}")
            print(f"   Indoor/contributor:     {n_indoor}")
            print(f"   Too far (>{snap_max_m}m):       {n_too_far}")
            print(f"   Duplicate panoramas:    {n_deduped}")
            print(f"   Surviving points:       {len(meta_df)}")

            # Build snap lookup
            surviving_ids = set(meta_df['point_id'])
            snap_lookup = dict(zip(
                meta_df['point_id'],
                zip(meta_df['lat'], meta_df['lon'],
                    meta_df['pano_id'], meta_df['snap_dist_m'])
            ))

            # Update points GeoDataFrame
            points_gdf_local = points_gdf_local[
                points_gdf_local['point_id'].isin(surviving_ids)
            ].copy()

            from shapely.geometry import Point as ShapelyPoint
            points_gdf_local['Latitude'] = points_gdf_local['point_id'].map(
                lambda pid: snap_lookup[pid][0])
            points_gdf_local['Longitude'] = points_gdf_local['point_id'].map(
                lambda pid: snap_lookup[pid][1])
            points_gdf_local['pano_id'] = points_gdf_local['point_id'].map(
                lambda pid: snap_lookup[pid][2])
            points_gdf_local['snap_dist_m'] = points_gdf_local['point_id'].map(
                lambda pid: round(snap_lookup[pid][3], 1))
            points_gdf_local['geometry'] = points_gdf_local.apply(
                lambda r: ShapelyPoint(r['Longitude'], r['Latitude']), axis=1)
            points_gdf_local = points_gdf_local.set_crs(
                "EPSG:4326", allow_override=True)

            # Save bearing into GeoPackage for downstream use
            points_gdf_local['bearing'] = points_gdf_local['point_id'].map(
                lambda pid: round(point_bearings.get(pid, 0.0), 1))

            # Save updated GeoPackage
            points_gdf_local.to_file(gpkg_path, layer="points", driver="GPKG")
            print(f"   GeoPackage updated with snapped points + bearings")

            # ==============================================
            # PHASE 3: Download with progress
            # ==============================================
            total_downloaded = 0
            total_skipped = 0
            total_na = 0
            n_points = len(points_gdf_local)
            t_start = time.time()

            print(f"\nPhase 3: Downloading images ({mode}) "
                  f"for {n_points} points...")

            progress_bar.max = n_points
            progress_bar.value = 0
            progress_bar.bar_style = 'success'
            progress_bar.description = 'Download:'

            for i, (idx, row) in enumerate(points_gdf_local.iterrows()):
                lat = row['Latitude']
                lon = row['Longitude']
                point_id = row['point_id']
                pano_id = row.get('pano_id', None)

                bearing = point_bearings.get(point_id, 0.0)
                headings = get_headings_for_mode(mode, bearing)

                for heading_val, heading_suffix in headings:
                    base_filename = f"{point_id}_{heading_suffix}"
                    expected_files = [
                        os.path.join(download_dir, f"{base_filename}.jpg"),
                        os.path.join(download_dir, f"{base_filename}_NA.jpg")
                    ]
                    if any(os.path.exists(f) for f in expected_files):
                        total_skipped += 1
                        continue
                    is_blank = download_streetview(
                        lat, lon, heading_val, pitch, fov,
                        download_dir, base_filename, api_key,
                        pano_id=pano_id,
                    )
                    if is_blank:
                        total_na += 1
                    else:
                        total_downloaded += 1

                progress_bar.value = i + 1
                elapsed = time.time() - t_start
                rate = (i + 1) / elapsed if elapsed > 0 else 0
                eta = (n_points - i - 1) / rate if rate > 0 else 0
                n_img_done = total_downloaded + total_na + total_skipped
                progress_label.value = (
                    f'<small>'
                    f'Points: {i+1}/{n_points} ({(i+1)/n_points*100:.0f}%) | '
                    f'Images: {n_img_done} '
                    f'({total_downloaded} new, {total_skipped} cached, '
                    f'{total_na} blank) | '
                    f'ETA: {int(eta//60)}m{int(eta%60):02d}s'
                    f'</small>'
                )

            n_per_point = len(headings)
            mode_label = {'4_views': '4 views', 'left_right': 'left+right',
                          'front_back': 'front+back'}[mode]
            elapsed_total = time.time() - t_start

            progress_bar.bar_style = ''
            progress_label.value = (
                f'<small><b>Complete!</b> {total_downloaded + total_na} '
                f'images in {int(elapsed_total//60)}m'
                f'{int(elapsed_total%60):02d}s</small>'
            )

            print(f"\nDownload complete! ({elapsed_total:.0f}s)")
            print(f"   Mode: {mode_label} ({n_per_point} images/point)")
            print(f"   New: {total_downloaded} | "
                  f"Cached: {total_skipped} | Blank: {total_na}")
            print(f"   Folder: {download_dir}")

            print(f"\nGenerating coverage map...")
            n_with, n_without = plot_coverage_map(
                gpkg_path, download_dir, case_study, mode, n_per_point
            )
            print(f"   With images: {n_with} | Without: {n_without}")

        except NameError:
            print("Error: Run Block 1 first to set the case study name.")
        except Exception as e:
            print(f"Error: {e}")
            progress_bar.bar_style = 'danger'
            raise

run_block2_button.on_click(run_block2)

display(widgets.VBox([
    widgets.HTML('<h3>Block 2 — Street View Batch Downloader</h3>'),
    api_key_widget,
    image_mode_widget,
    image_mode_help,
    pitch_widget,
    fov_widget,
    max_sub_widget,
    max_sub_help,
    snap_threshold_widget,
    snap_help,
    progress_box,
    run_block2_button,
    block2_output,
]))


In [2]:
# ===============================================
# BLOCK 3 — Install UVLM + Load VLM
# ===============================================
# Installs the UVLM package from GitHub (once per session).
# Select a model, precision, and device placement, then click "Load model".

!pip install -q git+https://github.com/perezjoan/UVLM.git

from uvlm import load_model
from uvlm.registry import MODEL_CHOICES
from uvlm.utils import get_hf_token

import ipywidgets as widgets
from IPython.display import display

use_token_widget = widgets.Checkbox(
    value=True,
    description="Use Hugging Face token (from Colab secrets)",
)
model_widget = widgets.Dropdown(options=list(MODEL_CHOICES.keys()), value="[Qwen]  Qwen2.5-VL 7B Instruct", description="Model")
precision_widget = widgets.Dropdown(
    options=[("4-bit (bnb, low VRAM)", "4bit"), ("8-bit (bnb)", "8bit"), ("FP16 / auto", "fp16")],
    value="4bit",
    description="Precision",
)
low_cpu_widget = widgets.Checkbox(value=True, description="low_cpu_mem_usage")
device_map_widget = widgets.Dropdown(
    options=[
        ("GPU only (cuda:0)", "cuda0"),
        ("Auto (accelerate decides)", "auto"),
        ("GPU + CPU offload", "offload"),
    ],
    value="auto",
    description="Placement",
)

load_button = widgets.Button(description="Load model", button_style="success")
status_html = widgets.HTML(value="")

model_ctx = None

def on_load(b):
    global model_ctx
    status_html.value = "<b>Loading model...</b>"
    try:
        hf_token = get_hf_token() if use_token_widget.value else None
        model_ctx = load_model(
            model_name=model_widget.value,
            precision=precision_widget.value,
            device_map=device_map_widget.value,
            low_cpu_mem_usage=low_cpu_widget.value,
            hf_token=hf_token,
        )
        status_html.value = (
            f"<b>✅ Model loaded</b><br>"
            f"⏱️ {model_ctx['load_time_minutes']:.2f} min<br>"
            f"🔥 {model_widget.value} on {model_ctx['gpu_name']} ({precision_widget.value})"
        )
    except Exception as e:
        status_html.value = f"<b>❌ Load failed</b><br><code>{e}</code>"

load_button.on_click(on_load)

display(widgets.VBox([
    use_token_widget, model_widget, precision_widget,
    low_cpu_widget, device_map_widget, load_button, status_html,
]))


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 154.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 140.6 MB/s eta 0:00:00


Loading [Qwen]  Qwen2.5-VL 32B Instruct (4bit) ...


preprocessor_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model-00017-of-00018.safetensors:   0%|          | 0.00/3.90G [00:00<?, ?B/s]

model-00018-of-00018.safetensors:   0%|          | 0.00/3.10G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/18 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

Device map: {'': 0}
Model loaded in 4.81 min on NVIDIA A100-SXM4-80GB (4bit)


In [3]:
# ===============================================
# BLOCK 4 — Drive + Task Configuration
# ===============================================
# Mount Google Drive, configure analysis tasks, and set generation parameters.
# The image folder is pre-filled from Block 1 (case study name).

from google.colab import drive
drive.mount('/content/drive')

from uvlm.prompts import TASK_TYPES, ADVANCED_REASONING_FORMATS, ADVANCED_REASONING_MAX_TOKENS, build_prompt
from transformers import AutoProcessor

import os
import ipywidgets as widgets
from IPython.display import display

# ------------------------------------------------
# Image folder (pre-filled from Block 1 if available)
# ------------------------------------------------
default_folder = ""
try:
    default_folder = f"SAGAI/StreetViewBatchDownload_{case_study_name.capitalize()}"
except NameError:
    pass

folder_widget = widgets.Text(
    value=default_folder,
    description="Image folder",
    placeholder="Folder name in MyDrive/",
    layout=widgets.Layout(width="80%"),
)

# ------------------------------------------------
# Role instruction (shared across all tasks)
# ------------------------------------------------
role_box = widgets.Textarea(
    value="",
    description="Role",
    placeholder="Optional: shared role instruction for all tasks",
    layout=widgets.Layout(width="80%", height="60px"),
)

# ------------------------------------------------
# Number of tasks
# ------------------------------------------------
n_tasks_widget = widgets.IntSlider(value=1, min=1, max=10, description="Tasks")

# ------------------------------------------------
# Dynamic task builder
# ------------------------------------------------
tasks_container = widgets.VBox()

def build_task_widgets(n):
    task_boxes = []
    for i in range(n):
        task_type = widgets.Dropdown(
            options=list(TASK_TYPES.keys()), value="numeric",
            description=f"Task {i+1} type", layout=widgets.Layout(width="40%"),
        )
        col_title = widgets.Text(
            value=f"task_{i+1}", description="Column name",
            layout=widgets.Layout(width="40%"),
        )
        task_prompt = widgets.Textarea(
            value="", description="Task prompt",
            placeholder="The question to ask about the image",
            layout=widgets.Layout(width="80%", height="60px"),
        )
        theory = widgets.Textarea(
            value="", description="Theory",
            placeholder="Optional: definitions, rules, edge cases",
            layout=widgets.Layout(width="80%", height="60px"),
        )
        fmt = widgets.Textarea(
            value="Answer with only one integer number, nothing else.",
            description="Format",
            layout=widgets.Layout(width="80%", height="40px"),
        )
        advanced = widgets.Checkbox(value=False, description="Advanced reasoning (CoT)")
        consensus = widgets.Checkbox(value=False, description="Consensus validation")
        consensus_runs = widgets.IntSlider(value=2, min=2, max=5, description="Runs")
        tolerance = widgets.FloatSlider(value=0.0, min=0.0, max=50.0, step=5.0,
            description="Tolerance %", layout=widgets.Layout(width="40%"))

        box = widgets.VBox([
            widgets.HTML(f"<hr><b>Task {i+1}</b>"),
            task_type, col_title, task_prompt, theory, fmt,
            advanced, consensus, consensus_runs, tolerance,
        ])
        task_boxes.append(box)
    tasks_container.children = task_boxes

build_task_widgets(1)
n_tasks_widget.observe(lambda change: build_task_widgets(change["new"]), names="value")

# ------------------------------------------------
# Qwen pixel settings (shown only for Qwen models)
# ------------------------------------------------
min_pixels_widget = widgets.IntText(value=200704, description="Min pixels")
max_pixels_widget = widgets.IntText(value=501760, description="Max pixels")
qwen_settings_box = widgets.VBox([
    widgets.HTML("<b>Qwen pixel settings</b>"),
    min_pixels_widget, max_pixels_widget,
])

if model_ctx is not None and model_ctx["backend"] == "qwen":
    qwen_settings_box.layout.display = "block"
    min_pixels_widget.value = model_ctx["qwen_min_pixels"]
    max_pixels_widget.value = model_ctx["qwen_max_pixels"]
else:
    qwen_settings_box.layout.display = "none"

# ------------------------------------------------
# Generation parameters
# ------------------------------------------------
do_sample_widget = widgets.Checkbox(value=True, description="do_sample")
temperature_widget = widgets.FloatSlider(value=0.3, min=0.0, max=1.5, step=0.05, description="Temperature")
top_p_widget = widgets.FloatSlider(value=0.9, min=0.0, max=1.0, step=0.05, description="Top-p")
max_tokens_widget = widgets.IntSlider(value=50, min=1, max=1500, step=10, description="Max tokens")
display_images_widget = widgets.Checkbox(value=False, description="Display images")
fixed_seed_widget = widgets.Checkbox(value=False, description="Fixed seed")
seed_value_widget = widgets.IntText(value=42, description="Seed value")

# ------------------------------------------------
# Apply button
# ------------------------------------------------
apply_button = widgets.Button(description="Apply paths + tasks + settings", button_style="success",
    layout=widgets.Layout(width="40%"))
status_out = widgets.Output()

# Global variables set by apply
image_folder = ""
output_path = ""
task_specs = []
max_new_tokens = 50
do_sample = True
temperature = 0.3
top_p = 0.9
generation_seed = None
display_images = False

def apply_settings(b):
    global image_folder, output_path, task_specs
    global max_new_tokens, do_sample, temperature, top_p, generation_seed, display_images

    with status_out:
        status_out.clear_output()

        # Paths
        root_path = "/content/drive/MyDrive"
        folder_name = folder_widget.value.strip()
        if not folder_name:
            print("❌ Please enter an image folder name.")
            return

        image_folder = os.path.join(root_path, folder_name)
        if not os.path.isdir(image_folder):
            print(f"❌ Folder not found: {image_folder}")
            return

        backend = model_ctx["backend"] if model_ctx else "unknown"
        out_name = f"Score_Analysis_{backend}.csv"
        output_path = os.path.join(image_folder, out_name)

        # Qwen pixel reload if needed
        if model_ctx and model_ctx["backend"] == "qwen":
            new_min = min_pixels_widget.value
            new_max = max_pixels_widget.value
            if new_min != model_ctx["qwen_min_pixels"] or new_max != model_ctx["qwen_max_pixels"]:
                print(f"🔄 Reloading Qwen processor with pixels [{new_min}, {new_max}]...")
                model_ctx["processor"] = AutoProcessor.from_pretrained(
                    model_ctx["model_id"],
                    min_pixels=new_min, max_pixels=new_max,
                )
                model_ctx["qwen_min_pixels"] = new_min
                model_ctx["qwen_max_pixels"] = new_max

        # Build task specs
        task_specs = []
        for box in tasks_container.children:
            children = box.children
            task_type_w = children[1]
            col_w = children[2]
            prompt_w = children[3]
            theory_w = children[4]
            fmt_w = children[5]
            adv_w = children[6]
            cons_w = children[7]
            runs_w = children[8]
            tol_w = children[9]

            role = role_box.value.strip()
            full_prompt = build_prompt(role, prompt_w.value.strip(), theory_w.value.strip(), fmt_w.value.strip())

            task_specs.append({
                "column": col_w.value.strip(),
                "prompt": full_prompt,
                "task_type": task_type_w.value,
                "advanced_reasoning": adv_w.value and task_type_w.value != "text",
                "consensus_enabled": cons_w.value and task_type_w.value != "text",
                "consensus_runs": runs_w.value,
                "numeric_tolerance": tol_w.value,
            })

        # Generation params
        max_new_tokens = max_tokens_widget.value
        do_sample = do_sample_widget.value
        temperature = temperature_widget.value
        top_p = top_p_widget.value
        generation_seed = seed_value_widget.value if fixed_seed_widget.value else None
        display_images = display_images_widget.value

        # Summary
        print(f"✅ Settings applied")
        print(f"📁 Image folder: {image_folder}")
        print(f"💾 Output: {output_path}")
        print(f"📝 Tasks: {len(task_specs)}")
        for s in task_specs:
            extra = []
            if s["advanced_reasoning"]: extra.append("reasoning")
            if s["consensus_enabled"]: extra.append(f"consensus×{s['consensus_runs']}")
            tag = f" [{', '.join(extra)}]" if extra else ""
            print(f"   • {s['column']} ({s['task_type']}){tag}")
        print(f"🎯 max_tokens={max_new_tokens}, do_sample={do_sample}, temp={temperature}, top_p={top_p}")

apply_button.on_click(apply_settings)

display(
    folder_widget, role_box, n_tasks_widget, tasks_container,
    qwen_settings_box, display_images_widget,
    do_sample_widget, temperature_widget, top_p_widget, max_tokens_widget,
    fixed_seed_widget, seed_value_widget,
    apply_button, status_out,
)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Text(value='SAGAI/StreetViewBatchDownload_Epb', description='Image folder', layout=Layout(width='80%'), placeh…

Textarea(value='', description='Role', layout=Layout(height='60px', width='80%'), placeholder='Optional: share…

IntSlider(value=1, description='Tasks', max=10, min=1)

Checkbox(value=False, description='Display images')

...

IntSlider(value=50, description='Max tokens', max=1500, min=1, step=10)

Checkbox(value=False, description='Fixed seed')

IntText(value=42, description='Seed value')

Button(button_style='success', description='Apply paths + tasks + settings', layout=Layout(width='40%'), style…

Output()

In [4]:
# ===============================================
# BLOCK 5 — Run Analysis (UVLM)
# ===============================================
# Processes all images using the official UVLM package.
# Results are saved to CSV with resume support.

from uvlm.batch import run_batch

if model_ctx is None:
    raise RuntimeError("❌ No model loaded. Run Block 3 first.")
if not task_specs:
    raise RuntimeError("❌ No tasks configured. Run Block 4 and click Apply first.")

df = run_batch(
    model_ctx=model_ctx,
    task_specs=task_specs,
    image_folder=image_folder,
    output_path=output_path,
    max_new_tokens=max_new_tokens,
    do_sample=do_sample,
    temperature=temperature,
    top_p=top_p,
    seed=generation_seed,
    display_images=display_images,
)

print(f"\n✅ Done! Results saved to {output_path}")
print(f"📊 {len(df)} images processed, {len(task_specs)} tasks each")


Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
      (raw: 6)

Processing: point_682_left.jpg
  sidewalk [boolean]: 1
      (raw: Yes)
  entrances [numeric]: 2
      (raw: 2)
  vegetation_type [numeric]: 4
      (raw: 4)

Processing: point_682_right.jpg
  sidewalk [boolean]: 0
      (raw: No.)
  entrances [numeric]: 0
      (raw: 0)
  vegetation_type [numeric]: 4
      (raw: 4)
  Checkpoint saved (7476 images processed)

Processing: point_683_left.jpg
  sidewalk [boolean]: 1
      (raw: Yes)
  entrances [numeric]: 0
      (raw: 0)
  vegetation_type [numeric]: 4
      (raw: 4)

Processing: point_683_right.jpg
  sidewalk [boolean]: 0
      (raw: No.)
  entrances [numeric]: 0
      (raw: 0)
  vegetation_type [numeric]: 4
      (raw: 4)

Processing: point_684_left.jpg
  sidewalk [boolean]: 1
      (raw: Yes)
  entrances [numeric]: 0
      (raw: 0)
  vegetation_type [numeric]: 4
      (raw: 4)
  Checkpoint saved (7479 images processed)

Processing: point_684_r

In [5]:
# ===============================================
# BLOCK 6 — Aggregation & Mapping
# ===============================================
# Aggregates image-level scores to point and street levels.
# ALL task columns from the CSV are spatially joined to the output
# GeoPackage (mean, sum, median, var + per-view-direction means),
# regardless of which indicator is selected for map display.
# The mapping selection (score column, aggregation, colormap)
# controls only the visual output, not what gets saved.
# View filter: aggregate all views, or left/right/front/back only.
# Optional interactive HTML maps (points / streets) saved to Drive.

!pip install -q folium

import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import os
import base64
import folium
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# ------------------------------------------------
# UI Widgets
# ------------------------------------------------
task_col_widget = widgets.Text(
    value='task_1',
    description='Score column',
    placeholder='Column name from the CSV (e.g., task_1)',
    layout=widgets.Layout(width='60%'),
)

aggregation_widget = widgets.Dropdown(
    options=[
        ('Average (mean)', 'average'),
        ('Sum (total)', 'sum'),
        ('Median', 'median'),
    ],
    value='average',
    description='Aggregation',
    layout=widgets.Layout(width='50%'),
)

view_filter_widget = widgets.Dropdown(
    options=[
        ('All views', 'all'),
        ('Left only', 'left'),
        ('Right only', 'right'),
        ('Front only', 'front'),
        ('Back only', 'back'),
        ('Left + Right', 'left_right'),
        ('Front + Back', 'front_back'),
    ],
    value='all',
    description='View filter',
    layout=widgets.Layout(width='50%'),
)

view_filter_help = widgets.HTML(
    value="<small><i>"
          "Select which views to include in aggregation. "
          "Useful for comparing scores between street sides (left vs right) "
          "or along-street vs cross-street views."
          "</i></small>"
)

colormap_widget = widgets.Dropdown(
    options=[
        ('Viridis (blue-green-yellow)', 'viridis'),
        ('Plasma (purple-orange-yellow)', 'plasma'),
        ('Inferno (black-red-yellow)', 'inferno'),
        ('Magma (black-purple-yellow)', 'magma'),
        ('Cividis (blue-yellow, colorblind-safe)', 'cividis'),
        ('RdYlGn (red-yellow-green)', 'RdYlGn'),
        ('Spectral (red-orange-green-blue)', 'Spectral'),
        ('Turbo (rainbow, high contrast)', 'turbo'),
        ('Coolwarm (blue-white-red, diverging)', 'coolwarm'),
        ('YlOrRd (yellow-orange-red)', 'YlOrRd'),
        ('RdBu (red-blue, diverging)', 'RdBu'),
    ],
    value='viridis',
    description='Color ramp',
    layout=widgets.Layout(width='55%'),
)

colormap_help = widgets.HTML(
    value="<small><i>"
          "Applied to both static and interactive maps. "
          "<b>Viridis/Plasma/Cividis</b>: perceptually uniform, "
          "colorblind-safe. "
          "<b>RdYlGn</b>: intuitive red=bad, green=good. "
          "<b>Turbo</b>: maximum visual contrast."
          "</i></small>"
)

run_block6_button = widgets.Button(
    description='Aggregate & Map', button_style='success'
)
build_points_map_btn = widgets.Button(
    description='Build Points HTML Map', button_style='info',
    layout=widgets.Layout(display='none'),
)
build_streets_map_btn = widgets.Button(
    description='Build Streets HTML Map', button_style='info',
    layout=widgets.Layout(display='none'),
)
block6_output = widgets.Output()
maps_output = widgets.Output()

# Store state for map builders
_block6_state = {}

# ------------------------------------------------
# Helpers
# ------------------------------------------------
def detect_image_mode(csv_df):
    filenames = csv_df['image_name'].tolist()
    suffixes = set()
    for fn in filenames:
        base = fn.replace('.jpg','').replace('.jpeg','').replace('.png','')
        base = base.replace('_NA', '')
        parts = base.split('_')
        if len(parts) >= 3:
            suffixes.add(parts[-1])
    if {'left','right','front','back'}.issubset(suffixes):
        return '4_views', 4
    elif {'left','right'}.issubset(suffixes):
        return 'left_right', 2
    elif {'front','back'}.issubset(suffixes):
        return 'front_back', 2
    return 'unknown', max(len(suffixes), 1)

def extract_base_id(image_name):
    base = image_name.replace('.jpg','').replace('.jpeg','').replace('.png','')
    base = base.replace('_NA', '')
    parts = base.split('_')
    if len(parts) >= 3:
        return '_'.join(parts[:-1])
    return base

def extract_view_suffix(image_name):
    base = image_name.replace('.jpg','').replace('.jpeg','').replace('.png','')
    base = base.replace('_NA', '')
    parts = base.split('_')
    if len(parts) >= 3:
        return parts[-1]
    return ''

def score_to_hex(score, vmin, vmax, cmap_name):
    if pd.isna(score):
        return '#cccccc'
    cmap = plt.get_cmap(cmap_name)
    val = max(0.0, min(1.0, (float(score) - vmin) / max(vmax - vmin, 0.001)))
    r, g, b, _ = cmap(val)
    return f'#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}'

# ------------------------------------------------
# Folium map builders
# ------------------------------------------------
def build_points_map(points_gdf_display, streets_gdf_display,
                     scores_df, score_col_name, pt_col,
                     download_dir, case_study, cmap_name):

    if 'base_id' not in scores_df.columns:
        scores_df = scores_df.copy()
        scores_df['base_id'] = scores_df['image_name'].apply(extract_base_id)

    valid_scores = points_gdf_display[pt_col].dropna()
    vmin = 0
    vmax = valid_scores.max() if len(valid_scores) > 0 else 1
    center_lat = points_gdf_display.geometry.y.mean()
    center_lon = points_gdf_display.geometry.x.mean()

    m = folium.Map(location=[center_lat, center_lon], zoom_start=14,
                   tiles='OpenStreetMap')

    if len(streets_gdf_display) > 0:
        folium.GeoJson(
            streets_gdf_display.to_json(),
            style_function=lambda x: {
                'color': '#888888', 'weight': 1.5, 'opacity': 0.5},
            name='Streets',
        ).add_to(m)

    for _, row in points_gdf_display.iterrows():
        pid = row['point_id']
        lat = row.geometry.y
        lon = row.geometry.x
        score_val = row.get(pt_col, None)
        street_id = row.get('street_id', '')
        color = score_to_hex(score_val, vmin, vmax, cmap_name)
        score_display = f"{float(score_val):.3f}" if pd.notna(score_val) else "N/A"

        imgs = scores_df[scores_df['base_id'] == pid]
        img_html_parts = []
        for _, irow in imgs.iterrows():
            fn = irow['image_name']
            img_score = irow.get(score_col_name, 'N/A')
            img_path = os.path.join(download_dir, fn)
            if not os.path.exists(img_path):
                img_path = os.path.join(download_dir, fn.replace('_NA',''))
            if os.path.exists(img_path):
                try:
                    with open(img_path, 'rb') as f:
                        img_b64 = base64.b64encode(f.read()).decode()
                    img_tag = (
                        f'<img src="data:image/jpeg;base64,{img_b64}" '
                        f'style="width:140px;height:140px;object-fit:cover;'
                        f'border-radius:3px;margin:2px;"/>')
                except Exception:
                    img_tag = '<span style="color:#999;">[error]</span>'
            else:
                img_tag = '<span style="color:#999;">[no file]</span>'
            s_display = img_score if str(img_score) != 'NA' else 'N/A'
            img_html_parts.append(
                f'<div style="display:inline-block;text-align:center;margin:3px;">'
                f'{img_tag}<br><small>{fn}</small><br>'
                f'<small>Score: <b>{s_display}</b></small></div>')
        images_html = ''.join(img_html_parts) if img_html_parts else '<i>No images</i>'

        popup_html = (
            f'<div style="min-width:320px;max-width:420px;font-family:Arial,sans-serif;">'
            f'<h4 style="margin:0 0 4px 0;">{pid}</h4>'
            f'<b>Score:</b> {score_display} | <b>Street:</b> {street_id}<br>'
            f'<b>Loc:</b> {lat:.5f}, {lon:.5f}'
            f'<hr style="margin:6px 0;">'
            f'<div style="display:flex;flex-wrap:wrap;justify-content:center;">'
            f'{images_html}</div></div>')

        folium.CircleMarker(
            location=[lat, lon], radius=6, color=color, fill=True,
            fill_color=color, fill_opacity=0.85, weight=1,
            popup=folium.Popup(popup_html, max_width=450),
        ).add_to(m)
    return m


def build_streets_map(streets_gdf_display, st_col, case_study, cmap_name):
    valid = streets_gdf_display[streets_gdf_display[st_col].notna()]
    vmin = 0
    vmax = valid[st_col].max() if len(valid) > 0 else 1
    center_lat = streets_gdf_display.geometry.centroid.y.mean()
    center_lon = streets_gdf_display.geometry.centroid.x.mean()

    m = folium.Map(location=[center_lat, center_lon], zoom_start=14,
                   tiles='OpenStreetMap')

    for _, row in streets_gdf_display.iterrows():
        score_val = row.get(st_col, None)
        sid = row.get('street_id', '')
        street_name = row.get('name', '') if pd.notna(row.get('name', None)) else ''
        highway_type = row.get('highway', '') if pd.notna(row.get('highway', None)) else ''
        color = score_to_hex(score_val, vmin, vmax, cmap_name)
        score_display = f"{float(score_val):.3f}" if pd.notna(score_val) else "N/A"
        weight = 4 if pd.notna(score_val) else 1.5
        opacity = 0.9 if pd.notna(score_val) else 0.3

        popup_html = (
            f'<div style="min-width:200px;font-family:Arial,sans-serif;">'
            f'<h4 style="margin:0 0 4px 0;">{sid}</h4>'
            f'<b>Name:</b> {street_name}<br>'
            f'<b>Type:</b> {highway_type}<br>'
            f'<b>Score:</b> {score_display}'
            f'</div>')

        geojson = folium.GeoJson(
            row.geometry.__geo_interface__,
            style_function=lambda x, c=color, w=weight, o=opacity: {
                'color': c, 'weight': w, 'opacity': o},
        )
        geojson.add_child(folium.Popup(popup_html, max_width=300))
        geojson.add_to(m)
    return m

# ------------------------------------------------
# Map build button callbacks
# ------------------------------------------------
def on_build_points_map(b):
    with maps_output:
        clear_output(wait=True)
        s = _block6_state
        if not s:
            print("Run Aggregate & Map first.")
            return
        print("Building points HTML map (this may take a moment)...")
        fmap = build_points_map(
            s['points_gdf'], s['streets_gdf'], s['filtered_df'],
            s['score_col'], s['pt_col'], s['download_dir'],
            s['case_study'], s['cmap'])
        path = f"{s['output_dir']}/{s['case_study']}_points_map.html"
        fmap.save(path)
        print(f"Points map saved!")
        print(f"   {path}")
        print(f"\nOpen it in your browser from Google Drive > SAGAI > Spatial_Results")

def on_build_streets_map(b):
    with maps_output:
        clear_output(wait=True)
        s = _block6_state
        if not s:
            print("Run Aggregate & Map first.")
            return
        print("Building streets HTML map...")
        fmap = build_streets_map(
            s['streets_gdf'], s['st_col'], s['case_study'], s['cmap'])
        path = f"{s['output_dir']}/{s['case_study']}_streets_map.html"
        fmap.save(path)
        print(f"Streets map saved!")
        print(f"   {path}")
        print(f"\nOpen it in your browser from Google Drive > SAGAI > Spatial_Results")

build_points_map_btn.on_click(on_build_points_map)
build_streets_map_btn.on_click(on_build_streets_map)

# ------------------------------------------------
# Main callback
# ------------------------------------------------
def run_block6(b):
    with block6_output:
        clear_output(wait=True)

        try:
            case_study = case_study_name
            score_col_name = task_col_widget.value.strip()
            agg_mode = aggregation_widget.value
            view_filter = view_filter_widget.value
            cmap_name = colormap_widget.value

            base_dir = '/content/drive/MyDrive/SAGAI'
            gpkg_path = f'{base_dir}/StreetSamples/{case_study}_osm.gpkg'
            download_dir = f'{base_dir}/StreetViewBatchDownload_{case_study.capitalize()}'

            csv_path = ''
            for search_dir in [download_dir, f'{base_dir}/Image_Analysis']:
                if os.path.exists(search_dir):
                    candidates = [f for f in os.listdir(search_dir)
                                 if f.endswith('.csv')]
                    if candidates:
                        csv_path = os.path.join(search_dir, candidates[0])
                        break

            if not csv_path or not os.path.exists(csv_path):
                print("CSV file not found in expected locations.")
                return

            output_dir = f'{base_dir}/Spatial_Results'
            os.makedirs(output_dir, exist_ok=True)
            output_gpkg = f'{output_dir}/{case_study}_osm_joined.gpkg'

            points_gdf_local = gpd.read_file(gpkg_path, layer='points')
            streets_gdf_local = gpd.read_file(gpkg_path, layer='streets')
            scores_df = pd.read_csv(csv_path, dtype=str)

            img_mode, n_images = detect_image_mode(scores_df)
            print(f"Case study: {case_study}")
            print(f"Image mode: {img_mode} ({n_images} images/point)")
            print(f"CSV: {os.path.basename(csv_path)}")
            print(f"Color ramp: {cmap_name}")

            if score_col_name not in scores_df.columns:
                avail = [c for c in scores_df.columns
                        if c not in ('image_name',)
                        and not c.endswith('_raw')
                        and not c.endswith('_reasoning')]
                print(f'Column "{score_col_name}" not found. '
                      f'Available: {", ".join(avail)}')
                return

            scores_df['base_id'] = scores_df['image_name'].apply(extract_base_id)
            scores_df['view'] = scores_df['image_name'].apply(extract_view_suffix)
            scores_df['score'] = pd.to_numeric(
                scores_df[score_col_name].replace("NA", pd.NA), errors='coerce')

            if view_filter == 'all':
                filtered_df = scores_df
                filter_label = 'all views'
            elif view_filter in ('left', 'right', 'front', 'back'):
                filtered_df = scores_df[scores_df['view'] == view_filter]
                filter_label = f'{view_filter} only'
            elif view_filter == 'left_right':
                filtered_df = scores_df[scores_df['view'].isin(['left','right'])]
                filter_label = 'left + right'
            elif view_filter == 'front_back':
                filtered_df = scores_df[scores_df['view'].isin(['front','back'])]
                filter_label = 'front + back'
            else:
                filtered_df = scores_df
                filter_label = 'all views'

            print(f"View filter: {filter_label} "
                  f"({len(filtered_df)}/{len(scores_df)} images)")

            all_base_ids = points_gdf_local[['point_id']].rename(
                columns={'point_id': 'base_id'})
            merged = all_base_ids.merge(
                filtered_df[['base_id','score']], on='base_id', how='left')
            point_stats = merged.groupby('base_id', dropna=False)['score'].agg(
                mean='mean', sum='sum', median='median', var='var'
            ).reset_index()
            points_gdf_local = points_gdf_local.merge(
                point_stats, left_on='point_id', right_on='base_id', how='left')
            if 'base_id' in points_gdf_local.columns:
                points_gdf_local.drop(columns='base_id', inplace=True)

            valid_pts = points_gdf_local[points_gdf_local['mean'].notna()]
            street_stats = valid_pts.groupby('street_id').agg(
                avg_score=('mean','mean'), total_score=('sum','sum'),
                median_score=('median','mean'), var_score=('var','mean')
            ).reset_index()
            streets_gdf_local = streets_gdf_local.merge(
                street_stats, on='street_id', how='left')

            points_gdf_local = points_gdf_local.to_crs("EPSG:4326")
            streets_gdf_local = streets_gdf_local.to_crs("EPSG:4326")

            if agg_mode == 'average':
                pt_col, st_col = 'mean', 'avg_score'
                vmax_pt = 1
            elif agg_mode == 'sum':
                pt_col, st_col = 'sum', 'total_score'
                v = points_gdf_local[points_gdf_local[pt_col].notna()]
                vmax_pt = v[pt_col].max() if len(v) > 0 else 1
            else:
                pt_col, st_col = 'median', 'median_score'
                vmax_pt = 1

            # ---- STATIC MAPS ----
            fig, axes = plt.subplots(1, 2, figsize=(18, 10))
            vmin = 0

            for ax_idx, (ax, col, gdf, geom_type) in enumerate([
                (axes[0], pt_col, points_gdf_local, 'points'),
                (axes[1], st_col, streets_gdf_local, 'streets'),
            ]):
                valid = gdf[gdf[col].notna()]
                missing = gdf[gdf[col].isna()]
                vmax = vmax_pt if ax_idx == 0 else (
                    vmax_pt if agg_mode == 'average' else (
                        valid[col].max() if len(valid) > 0 else 1))

                if geom_type == 'points':
                    if len(valid) > 0:
                        valid.plot(ax=ax, column=col, cmap=cmap_name,
                                   markersize=50, legend=False,
                                   vmin=vmin, vmax=vmax, alpha=0.9)
                    if len(missing) > 0:
                        missing.plot(ax=ax, color='lightgrey', markersize=50)
                else:
                    if len(valid) > 0:
                        valid.plot(ax=ax, column=col, cmap=cmap_name,
                                   linewidth=3, legend=False,
                                   vmin=vmin, vmax=vmax)
                    if len(missing) > 0:
                        missing.plot(ax=ax, color='lightgrey', linewidth=3)

                title_type = 'Point' if geom_type == 'points' else 'Street'
                ax.set_title(f'{agg_mode.capitalize()} Score per {title_type} '
                             f'({filter_label})', fontsize=14)
                ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
                ax.tick_params(axis='x', rotation=45); ax.grid(True)
                sm = mpl.cm.ScalarMappable(
                    cmap=cmap_name,
                    norm=mpl.colors.Normalize(vmin=vmin, vmax=vmax))
                sm.set_array([])
                plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.04)

            plt.tight_layout(); plt.show()

            # Save GeoPackage
            points_gdf_local.to_file(output_gpkg, layer='points', driver='GPKG')
            streets_gdf_local.to_file(output_gpkg, layer='streets', driver='GPKG')
            print(f"\nGeoPackage: {output_gpkg}")

            # Store state for optional map building
            _block6_state.update({
                'points_gdf': points_gdf_local,
                'streets_gdf': streets_gdf_local,
                'filtered_df': filtered_df,
                'score_col': score_col_name,
                'pt_col': pt_col,
                'st_col': st_col,
                'download_dir': download_dir,
                'output_dir': output_dir,
                'case_study': case_study,
                'cmap': cmap_name,
            })

            # Show map buttons
            build_points_map_btn.layout.display = ''
            build_streets_map_btn.layout.display = ''

            print(f"\nAggregation complete!")
            print(f"   View filter: {filter_label}")
            print(f"   Score column: {score_col_name} | "
                  f"Aggregation: {agg_mode} | Color ramp: {cmap_name}")
            print(f"\nOptionally build interactive HTML maps "
                  f"using the buttons below.")

        except NameError:
            print("Error: Run Block 1 first to set the case study name.")
        except Exception as e:
            print(f"Error: {e}")
            raise

run_block6_button.on_click(run_block6)

display(widgets.VBox([
    widgets.HTML('<h3>Block 6 — Aggregation & Mapping</h3>'),
    task_col_widget,
    aggregation_widget,
    view_filter_widget,
    view_filter_help,
    colormap_widget,
    colormap_help,
    run_block6_button,
    block6_output,
    widgets.HBox([build_points_map_btn, build_streets_map_btn]),
    maps_output,
]))
